# HTML Navigator

In [ ]:
%load_ext autoreload
%autoreload 2

import tiktoken
import random, os
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

# Custom
import sys
sys.path.insert(1, '../')
from html2text import clean
from navigator import NavigatorHTML
from utils import format_path


In [ ]:
html = "../../data/wpcd/wp/"
paths = "../../data/wikispeedia_paths-and-graph/paths_finished.tsv"

## 0. Data preparation

In [ ]:
if os.path.exists(paths) and os.path.exists(html):
    print("The required data is avaiable.")
else:
    print("The required data is not avaiable. Please follow the instructions below to get them.")

Downloading the data [instructions](../data/README.md).

## 1. Cleaning

In [ ]:
article = html + "/u/United_States.htm"

with open(article, "r", encoding='unicode_escape') as f:
        doc = BeautifulSoup(f, "html.parser")
body = doc.find("div", {"id" : "bodyContent"})

text = clean(body, "None", "None")
print(text)

## 2. Navigation

In [ ]:
nav = NavigatorHTML(pages_path=html, token_window=300, overlap=0.1)
nav.set_curr_doc("United_States")
print(nav.window_text)

In [ ]:
nav.move_down()
print(nav.window_text)
print("Links in text :\n{}".format(nav.window_links))

In [ ]:
nav.move_up()
nav.move_up()
nav.move_up()
print(nav.window_text)

## 3. Traversing through a path

In [ ]:
def travel_path(path, verbose=False):
    path = format_path(path)
    if verbose:
        print("Full path :\n"+"->".join(path) + "\n")

    path_position = 0
    current_page = path[path_position]
    target_page = path[path_position+1]
    final_page = path[-1]

    nav = NavigatorHTML(pages_path=html)
    nav.set_curr_doc(current_page) 

    while nav.current_page != final_page:

        # In case you gotta turn back
        if target_page == "<":
            if verbose:
                print("{} <- {}".format(path[path_position-1], current_page))
            path_position -=1
            del path[path_position + 1]
            del path[path_position + 1]
            current_page = path[path_position]
            target_page = path[path_position + 1]
            nav.set_curr_doc(current_page)

        # In case target page is in window text
        if nav.check_target(target_page=target_page):
            if verbose:
                print("{} -> {}".format(current_page, target_page))
            path_position +=1
            if path_position >= len(path)-1:# reached final page
                break
            else:
                current_page = path[path_position]
                target_page = path[path_position + 1]
        
        # In case target page is not in the window text move down
        else:
            nav.move_down()

In [ ]:
df = pd.read_csv(paths, sep="\t", skiprows=16, names = ["hashedIpAddress", "timestamp", "durationInSec", "path", "rating"])

path = df.iloc[random.randint(0, len(df)-1)]["path"]
travel_path(path, verbose=True)

In [ ]:
path = "14th_century;Europe;<;England;<;Christianity;Anglicanism;United_States;Walt_Whitman;<;Poetry_of_the_United_States;<;<;<;French_Revolution;Age_of_Enlightenment;David_Hume;Philosophy;Fyodor_Dostoevsky;<;<;<;<;Constitutional_monarchy;Government;Anarchism;Henry_David_Thoreau"
travel_path(path, verbose=True)